# 05 — Reliability and Termination

Failure engineering starts by distinguishing poor output from failed execution. This deterministic graph applies different policies and proves that every recovery path is bounded.

## Semantic and runtime failures

| Failure | Examples | Policy |
| --- | --- | --- |
| Semantic | Low quality, insufficient evidence, failed evaluation | Feedback and improve |
| Transient runtime | Timeout, rate limit, temporary network error | Bounded retry and backoff |
| Permanent runtime | Permission denied, invalid credentials, unavailable capability | Fallback, abort, or escalate |

A semantic iteration must change evidence or strategy. A transient retry may repeat after time or dependency state changes. A permanent error should not be retried blindly.

## Recovery topology

```mermaid
flowchart TD
    accTitle: Reliability Policy Graph
    accDescr: Execution errors route to bounded runtime retry or fallback, while successful execution is assessed and low quality routes to bounded semantic improvement.

    execute[Execute] --> runtime_gate{Runtime status}
    runtime_gate -->|Success| assess[Assess quality]
    runtime_gate -->|Transient and budget remains| backoff[Backoff]
    backoff --> execute
    runtime_gate -->|Permanent or exhausted| fallback([Fallback])
    assess --> quality_gate{Quality policy}
    quality_gate -->|Pass| complete([Complete])
    quality_gate -->|Fail and budget remains| improve[Change strategy]
    improve --> execute
    quality_gate -->|Exhausted| fallback
```

## State carries separate budgets

Runtime retries and semantic attempts are different counters. The graph also records failure class and termination reason so the final state explains why execution ended.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

MAX_RUNTIME_RETRIES = 2
MAX_SEMANTIC_ATTEMPTS = 3
QUALITY_THRESHOLD = 0.8


class ReliableState(TypedDict, total=False):
    task: str
    transient_failures_remaining: int
    permanent_error: bool
    runtime_retries: int
    semantic_attempts: int
    quality_seed: float
    quality: float
    strategy: str
    status: str
    failure_kind: str
    result: str
    final_answer: str
    termination_reason: str
    trace: Annotated[list[str], operator.add]

## Execution classifies runtime outcomes

The node simulates provider behavior without sleeping or using a network. It returns structured status rather than hiding retry logic inside the operation.

In [2]:
def execute(state: ReliableState) -> dict:
    if state.get("permanent_error", False):
        return {
            "status": "error",
            "failure_kind": "permanent",
            "trace": ["execute:permission_denied"],
        }

    remaining = state.get("transient_failures_remaining", 0)
    if remaining > 0:
        return {
            "status": "error",
            "failure_kind": "transient",
            "transient_failures_remaining": remaining - 1,
            "trace": ["execute:timeout"],
        }

    strategy = state.get("strategy", "initial strategy")
    return {
        "status": "ok",
        "failure_kind": "none",
        "result": f"Result for {state['task']} using {strategy}",
        "trace": ["execute:ok"],
    }

## Runtime recovery is bounded

A transient failure may use retry and backoff. This lesson records backoff without delaying the notebook. A production implementation would honor dependency guidance and add jitter.

In [3]:
def route_runtime(state: ReliableState) -> str:
    if state["status"] == "ok":
        return "assess"
    if state["failure_kind"] == "transient" and state.get("runtime_retries", 0) < MAX_RUNTIME_RETRIES:
        return "backoff"
    return "fallback"


def backoff(state: ReliableState) -> dict:
    retry = state.get("runtime_retries", 0) + 1
    return {"runtime_retries": retry, "trace": [f"backoff:retry={retry}"]}

## Semantic recovery changes strategy

Assessment computes quality. The quality router applies the threshold and semantic-attempt budget. Improvement changes the next execution strategy.

In [4]:
def assess(state: ReliableState) -> dict:
    attempts = state.get("semantic_attempts", 0) + 1
    quality = min(state.get("quality_seed", 0.55) + 0.25 * (attempts - 1), 1.0)
    return {"semantic_attempts": attempts, "quality": quality, "trace": [f"assess:{quality:.2f}"]}


def route_quality(state: ReliableState) -> str:
    if state["quality"] >= QUALITY_THRESHOLD:
        return "complete"
    if state["semantic_attempts"] >= MAX_SEMANTIC_ATTEMPTS:
        return "fallback"
    return "improve"


def improve(state: ReliableState) -> dict:
    return {
        "strategy": f"revised after semantic attempt {state['semantic_attempts']}",
        "trace": ["improve:strategy_changed"],
    }

In [5]:
def complete(state: ReliableState) -> dict:
    return {
        "final_answer": state["result"],
        "termination_reason": "quality_reached",
        "trace": ["complete"],
    }


def fallback(state: ReliableState) -> dict:
    if state.get("status") == "error":
        reason = (
            "permanent_error"
            if state.get("failure_kind") == "permanent"
            else "runtime_retry_budget_exhausted"
        )
    else:
        reason = "semantic_attempt_budget_exhausted"
    return {"termination_reason": reason, "trace": [f"fallback:{reason}"]}

## Compile the policies as visible edges

Runtime and semantic recovery have different loop edges and different counters. Both converge on explicit completion or fallback nodes.

In [6]:
builder = StateGraph(ReliableState)
for name, node in {
    "execute": execute,
    "backoff": backoff,
    "assess": assess,
    "improve": improve,
    "complete": complete,
    "fallback": fallback,
}.items():
    builder.add_node(name, node)

builder.add_edge(START, "execute")
builder.add_conditional_edges("execute", route_runtime, {"assess": "assess", "backoff": "backoff", "fallback": "fallback"})
builder.add_edge("backoff", "execute")
builder.add_conditional_edges("assess", route_quality, {"complete": "complete", "improve": "improve", "fallback": "fallback"})
builder.add_edge("improve", "execute")
builder.add_edge("complete", END)
builder.add_edge("fallback", END)
graph = builder.compile()

## Verify recovery and permanent failure

The first scenario recovers from one timeout and one low-quality assessment. The second stops immediately on a permanent permission failure.

In [7]:
recovered = graph.invoke({
    "task": "research a market change",
    "transient_failures_remaining": 1,
    "runtime_retries": 0,
    "semantic_attempts": 0,
    "quality_seed": 0.55,
    "trace": [],
})
assert recovered["runtime_retries"] == 1
assert recovered["semantic_attempts"] == 2
assert recovered["termination_reason"] == "quality_reached"
print(recovered["trace"])

permanent = graph.invoke({
    "task": "access a protected tool",
    "permanent_error": True,
    "runtime_retries": 0,
    "semantic_attempts": 0,
    "trace": [],
})
assert permanent["runtime_retries"] == 0
assert permanent["termination_reason"] == "permanent_error"
print(permanent["trace"])

['execute:timeout', 'backoff:retry=1', 'execute:ok', 'assess:0.55', 'improve:strategy_changed', 'execute:ok', 'assess:0.80', 'complete']
['execute:permission_denied', 'fallback:permanent_error']


## Beyond attempt counts

Production graphs may also enforce token budgets, cost budgets, node timeouts, whole-run deadlines, cancellation, and fallbacks. Keep these exact policies outside the LLM. Make side effects idempotent so a retry or resumed checkpoint cannot repeat an irreversible action.

The learning sequence is complete: deterministic control, explicit state, routing, feedback, parallelism, reliability, and termination. Continue with the pattern library and research-agent example.